# LlamaIndex와 AgentCore Memory - 법률 문서 Analyzer(단기 메모리)

## 소개

이 Notebook에서는 Amazon Bedrock AgentCore Memory 기능을 LlamaIndex와 통합하여 법률 문서 Analyzer를 만드는 방법을 살펴봅니다. 하나의 법률 분석 세션 안에서 **단기 메모리**를 유지하여 법률 검토 전반에 걸쳐 계약 조항, 판례, 규정 준수 문제를 기억하도록 하는 데 중점을 둡니다.

## 아키텍처 개요

![LlamaIndex AgentCore Short-Term Memory Architecture](LlamaIndex-AgentCore-STM-Arch.png)

## 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화 메모리                                                |
| Agent 사용 사례       | 법률 문서 Analyzer                                                          |
| Agentic Framework   | LlamaIndex                                                                       |
| LLM 모델           | Anthropic Claude 3.7 Sonnet                                                  |
| 튜토리얼 구성 요소 | AgentCore Short-term Memory, LlamaIndex Agent, 법률 분석 도구             |
| 예제 난이도  | 초급                                                                         |

다음 내용을 학습합니다.
- 법률 문서 분석을 위한 AgentCore Memory 생성
- 법률 workflow에 LlamaIndex 기본 메모리 통합 사용
- 계약 분석을 위한 법률 전용 도구 구축
- 단일 분석 세션 내에서 법률 컨텍스트 유지
- 메모리 경계 및 세션 격리 테스트

## 시나리오 배경

이 예제에서는 변호사가 단일 법률 검토 세션 안에서 계약을 분석하고 법률 문제를 추적하며 규정 준수 요구 사항을 관리하도록 돕는 "법률 문서 Analyzer"를 만듭니다. Analyzer는 AgentCore Memory를 사용하여 분석 전반에 걸쳐 계약 조항, 위험 평가, 판례, 규정 준수 문제에 관한 컨텍스트를 유지합니다.

## 사전 요구 사항

- Python 3.10 이상
- 적절한 권한이 있는 AWS 계정
- AgentCore Memory 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델에 대한 액세스

## 1단계: 종속성 설치 및 설정

In [ ]:
# 필요한 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3

In [ ]:
# 필요한 구성 요소 가져오기
from bedrock_agentcore.memory import MemoryClient
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

## 2단계: AgentCore Memory 구성

법률 Analyzer에서 사용할 AgentCore Memory 리소스를 생성하거나 가져옵니다.

In [ ]:
# AgentCore Memory 리소스 생성
region = os.getenv("AWS_REGION", "us-east-1")
client = MemoryClient(region_name=region)

try:
    response = client.create_memory_and_wait(
        name=f"LegalAnalyzerShortTerm_{int(datetime.now().timestamp())}",
        description="Legal document analyzer short-term memory for single session context",
        strategies=[],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10,
    )
    memory_id = response["id"]
    print(f"✅ Created AgentCore Memory: {memory_id}")
except Exception as e:
    print(f"❌ Error creating memory: {e}")
    memory_id = "your-memory-id-here"  # 기존 Memory ID로 교체

## 3단계: 법률 분석 도구 구현

법률 문서 분석을 위한 전문 도구를 정의합니다.

In [ ]:
def analyze_contract_clause(clause_text: str, clause_type: str, risk_level: str) -> str:
    """Analyze a contract clause and assess its risk level"""
    print(f"⚖️ Analyzed {clause_type} clause (Risk: {risk_level})")
    return f"Analyzed {clause_type} clause with {risk_level} risk assessment"


def track_legal_issue(issue: str, priority: str, status: str) -> str:
    """Track legal issue with priority and status"""
    print(f"📋 Tracking legal issue: {issue} ({priority} priority, {status})")
    return f"Now tracking legal issue: {issue}"


def save_legal_precedent(case_name: str, jurisdiction: str, relevance: str) -> str:
    """Save legal precedent with jurisdiction and relevance"""
    print(f"📚 Saved legal precedent: {case_name} ({jurisdiction})")
    return f"Saved legal precedent: {case_name}"


def flag_compliance_issue(regulation: str, violation_type: str, severity: str) -> str:
    """Flag compliance issue with regulation and severity"""
    print(f"🚨 Flagged {severity} compliance issue: {regulation}")
    return f"Flagged compliance issue: {regulation}"


# Agent용 도구 객체 생성
legal_tools = [
    FunctionTool.from_defaults(fn=analyze_contract_clause),
    FunctionTool.from_defaults(fn=track_legal_issue),
    FunctionTool.from_defaults(fn=save_legal_precedent),
    FunctionTool.from_defaults(fn=flag_compliance_issue),
]

## 4단계: LlamaIndex Agent 구현

단기 메모리 컨텍스트를 사용하는 법률 Analyzer Agent를 생성합니다.

In [ ]:
# 단기 메모리 구성(단일 세션)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

# 단일 세션용 메모리 컨텍스트 생성
context = AgentCoreMemoryContext(
    actor_id="legal-analyst",
    memory_id=memory_id,
    session_id="legal-analysis-session-today",  # 전체 과정에서 동일한 세션 사용
    namespace="/legal-analysis/",
)

# AgentCore Memory 및 LLM 초기화
agentcore_memory = AgentCoreMemory(context=context)
llm = BedrockConverse(model=MODEL_ID)

# 법률 Analyzer Agent 생성
legal_agent = FunctionAgent(tools=legal_tools, llm=llm, verbose=True)

print("✅ Legal Document Analyzer with short-term memory is ready!")

## 5단계: 단기 메모리 기능 테스트

종합 계약 분석 세션을 통해 법률 Analyzer의 단기 메모리를 테스트해 보겠습니다.

### 테스트 1: 사례 설정 및 초기화

In [ ]:
# 세부 컨텍스트로 법률 분석 세션 초기화
response = await legal_agent.run(
    "I'm Attorney Maria Johnson from Johnson & Associates, analyzing a $5M software licensing agreement "
    "between TechCorp (licensor) and DataSoft Inc (licensee). Track this as 'Software License Review' "
    "with critical priority and active status. Contract value: $5M over 3 years.",
    memory=agentcore_memory,
)

print("🎯 Case Setup:")
print(response)

### 테스트 2: 계약 조항 분석

In [ ]:
# 구체적인 조건이 포함된 책임 조항 분석
response = await legal_agent.run(
    "Analyze this liability clause: 'TechCorp's total liability shall not exceed $50,000 for any direct damages "
    "and excludes all indirect, consequential, or punitive damages.' This is a 'Liability Limitation' clause "
    "with 'High' risk level due to low cap vs contract value.",
    memory=agentcore_memory,
)

print("⚖️ Liability Clause Analysis:")
print(response)

In [ ]:
# 통지 기간이 포함된 해지 조항 분석
response = await legal_agent.run(
    "Analyze termination clause: 'Either party may terminate with 90 days written notice. "
    "TechCorp may terminate immediately for material breach or non-payment exceeding 30 days.' "
    "This is a 'Termination' clause with 'Medium' risk level.",
    memory=agentcore_memory,
)

print("📋 Termination Clause Analysis:")
print(response)

### 테스트 3: 계약 컨텍스트 회상

In [ ]:
# 계약 컨텍스트 및 위험 평가 회상 테스트
response = await legal_agent.run(
    "What contract am I analyzing? Who are the parties, what's the value, and what's my assessment of the liability cap?",
    memory=agentcore_memory,
)

print("🧠 Contract Context Recall:")
print(response)
print("\n✅ Expected: TechCorp/DataSoft, $5M contract, $50K liability cap (high risk)")

### 테스트 4: 세부 조항 회상

In [ ]:
# 특정 조항 세부 정보 회상 테스트
response = await legal_agent.run(
    "What are the exact termination notice periods I found? What triggers immediate termination?",
    memory=agentcore_memory,
)

print("📋 Termination Details Recall:")
print(response)
print("\n✅ Expected: 90 days notice, immediate for breach or 30+ day non-payment")

### 테스트 5: 판례 통합

In [ ]:
# 사례 세부 정보와 관련 판례 저장
response = await legal_agent.run(
    "Save legal precedent: 'TechSoft Inc. v. MegaCorp' from 'Delaware Superior Court' with 'Critical' relevance. "
    "This case established that liability caps below 1% of contract value are unconscionable in software licensing.",
    memory=agentcore_memory,
)

print("📚 Legal Precedent Saved:")
print(response)

### 테스트 6: 위험 평가 추론

In [ ]:
# 위험 평가 추론 테스트
response = await legal_agent.run(
    "Why did I assess the liability clause as high risk? What's the mathematical relationship between the cap and contract value?",
    memory=agentcore_memory,
)

print("🤔 Risk Assessment Reasoning:")
print(response)
print("\n✅ Expected: $50K cap vs $5M contract = 1% ratio, high risk due to low percentage")

### 테스트 7: 규정 준수 문제 표시

In [ ]:
# 규제 세부 정보와 규정 준수 문제 표시
response = await legal_agent.run(
    "Flag compliance issue: 'GDPR Article 82 Data Protection' violation type 'Inadequate Liability Coverage' "
    "with 'Critical' severity. The $50K cap is insufficient for potential GDPR fines up to 4% of annual revenue.",
    memory=agentcore_memory,
)

print("🚨 GDPR Compliance Issue:")
print(response)

### 테스트 8: 판례 적용

In [ ]:
# 현재 사례에 판례 적용 테스트
response = await legal_agent.run(
    "How does the TechSoft v. MegaCorp precedent apply to my current contract analysis? "
    "What does it suggest about the liability clause?",
    memory=agentcore_memory,
)

print("⚖️ Precedent Application:")
print(response)
print("\n✅ Expected: Both have ~1% liability caps, precedent suggests unconscionability")

### 테스트 9: 종합 위험 평가

In [ ]:
# 종합 위험 평가 질의
response = await legal_agent.run(
    "Provide a comprehensive risk assessment for DataSoft: What are all the risks I've identified, "
    "their severity levels, and supporting precedents?",
    memory=agentcore_memory,
)

print("📊 Comprehensive Risk Assessment:")
print(response)
print("\n✅ Expected: High risk liability cap, GDPR compliance issues, TechSoft precedent support")

## 6단계: 세션 경계 테스트

별도의 세션을 생성하여 단기 메모리의 경계를 테스트해 보겠습니다.

In [ ]:
# 별도의 세션 컨텍스트 생성
new_session_context = AgentCoreMemoryContext(
    actor_id="legal-analyst",
    memory_id=memory_id,
    session_id="different-legal-session",  # 서로 다른 세션 ID
    namespace="/legal-analysis/",
)

new_session_memory = AgentCoreMemory(context=new_session_context)

# 메모리 격리 테스트
response = await legal_agent.run(
    "What contracts am I analyzing? What liability caps and compliance issues have I found?",
    memory=new_session_memory,
)

print("🚧 Session Boundary Test (Different Session):")
print(response)
print("\n✅ Expected: Limited or no recall from previous session (short-term memory boundary)")

In [ ]:
# 지속성을 검증하기 위해 원래 세션으로 복귀
response = await legal_agent.run(
    "Back in my original session - what was the exact liability cap amount and GDPR compliance issue I identified?",
    memory=agentcore_memory,  # 원래 세션 메모리
)

print("🔄 Original Session Return:")
print(response)
print("\n✅ Expected: Full recall of $50K cap, GDPR Article 82 issue")

## 🧪 자동 테스트 검증
다음 셀을 실행하여 메모리 통합이 올바르게 작동하는지 검증합니다.

In [ ]:
# 검증 함수를 인라인으로 정의
class TestValidator:
    def __init__(self):
        self.results = {}

    def validate_memory_recall(self, response):
        """에이전트가 세션 앞부분의 정보를 기억하는지 확인합니다."""
        # 실질적인 응답인지 확인("I don't know"만 있는 응답 제외)
        has_content = len(response) > 50
        # 메모리 관련 표현 확인
        has_memory_indicators = any(
            word in response.lower()
            for word in [
                "earlier",
                "mentioned",
                "discussed",
                "previously",
                "you",
                "we",
                "our",
            ]
        )
        return "✅ PASS" if (has_content and has_memory_indicators) else "❌ FAIL"

    def validate_session_memory(self, response):
        """에이전트가 세션 내 컨텍스트를 유지하는지 확인합니다."""
        has_memory_content = len(response) > 100 and any(
            word in response.lower()
            for word in [
                "previous",
                "earlier",
                "mentioned",
                "discussed",
                "before",
                "already",
            ]
        )
        return "✅ PASS" if has_memory_content else "❌ FAIL"

    def validate_cross_reference(self, response):
        """에이전트가 현재 질의를 이전 컨텍스트와 연결할 수 있는지 확인합니다."""
        # 연결 표현 확인
        connecting_words = [
            "relate",
            "connection",
            "previous",
            "earlier",
            "discussed",
            "mentioned",
            "context",
            "based on",
            "as we",
            "as i",
        ]
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "✅ PASS" if (has_connection and has_substance) else "❌ FAIL"

    def run_validation_summary(self, test_results):
        print("🧪 COMPREHENSIVE TEST VALIDATION SUMMARY")
        print("=" * 60)

        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "PASS" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0

        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")

        print("=" * 60)
        print(f"📊 Overall Pass Rate: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")

        if pass_rate >= 80:
            print("✅ EXCELLENT: Memory integration working correctly!")
        elif pass_rate >= 60:
            print("⚠️  GOOD: Most memory features working, some issues to investigate")
        else:
            print("❌ NEEDS ATTENTION: Memory integration has significant issues")

        return pass_rate


validator = TestValidator()
print("✅ Validation functions loaded!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상 - Agent가 논의한 내용을 기억하는가?
response1 = await legal_agent.run("What have we discussed so far in this session?", memory=agentcore_memory)
test_results["Memory Recall"] = validator.validate_memory_recall(str(response1))
print(f"Response 1 length: {len(str(response1))} chars")

# 테스트 2: 세션 메모리 - Agent가 컨텍스트를 유지하는가?
response2 = await legal_agent.run("What did we talk about earlier?", memory=agentcore_memory)
test_results["Session Memory"] = validator.validate_session_memory(str(response2))
print(f"Response 2 length: {len(str(response2))} chars")

# 테스트 3: 상호 참조 기능 - Agent가 이전 컨텍스트와 연결할 수 있는가?
response3 = await legal_agent.run("How does this relate to what we discussed before?", memory=agentcore_memory)
test_results["Cross Reference"] = validator.validate_cross_reference(str(response3))
print(f"Response 3 length: {len(str(response3))} chars")

# 결과 표시
validator.run_validation_summary(test_results)

## 요약

이 Notebook에서는 다음 내용을 구현했습니다.

✅ **단기 메모리 통합**: LlamaIndex에서 AgentCore Memory를 사용하여 세션 범위의 법률 분석 수행

✅ **법률 전용 도구**: 계약 조항 분석, 판례 관리, 규정 준수 추적

✅ **컨텍스트 기반 법률 분석**: Analyzer가 계약 세부 정보, 위험 평가, 판례를 기억

✅ **위험 평가 추론**: 책임 한도를 계약 금액 및 법률 판례와 연결

✅ **세션 경계**: 서로 다른 법률 분석 세션 간의 메모리 격리

✅ **규정 준수 관리**: 규제 문제와 심각도 추적

법률 문서 Analyzer는 단기 메모리를 통해 하나의 세션 안에서 종합적인 계약 분석을 수행하는 동시에 서로 다른 법률 사안 사이의 경계를 명확히 유지하는 방법을 보여 줍니다.

## 리소스 정리

이 Notebook에서 사용한 리소스를 정리하기 위해 Memory를 삭제합니다.

In [ ]:
# AgentCore Memory 리소스 정리
try:
    client.delete_memory(memory_id)
    print(f"✅ Successfully deleted memory: {memory_id}")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")